# Семинар 1

## План ноутбука

1. Установка `PyTorch`
1. Введение в `PyTorch`
1. Полносвязные слои и функции активации в `PyTorch`
1. Градиентный спуск своими руками

## Установка `PyTorch`

Мы будем использовать библиотеку для глубинного обучения `PyTorch`, ее можно не устанавливать, можно пользоваться сайтами [Kaggle](kaggle.com) и [Google Colab](colab.research.google.com/) для обучения в облаке. 

Локально обычно создают изолированное окружение при помощи [anaconda](https://www.anaconda.com/docs/getting-started/anaconda/install), [micromamba](https://mamba.readthedocs.io/en/latest/user_guide/micromamba.html), [uv](https://docs.astral.sh/uv/getting-started/installation/). И в него ставят необходимые пакеты.

Команду для установки `PyTorch` можно посмотреть на [сайте](https://pytorch.org/get-started/locally/).

Пререквизит для работы с видеокартой от Nvidia - нужно поставить CUDA, это инструмент от компании Nvidia, который позволяет ускорять вычисления на их же ГПУ. Чтобы поставить себе на машину все правильно воспользуйтесь этим [гайдом](https://docs.nvidia.com/cuda/cuda-installation-guide-linux/index.html) от Nvidia.

Для прохождения курса будет достаточно каггла с коллабом. Если у вас мак, то удобно использовать mps. Для него дополнительно никаких драйверов ставить не надо.

## Введение в `PyTorch`

### Тензоры

Тензоры — это специализированная структура данных, по сути это массивы и матрицы. Тензоры очень похожи на массивы в numpy, так что, если у вас хорошо с numpy, то разобраться в PyTorch тензорах будет очень просто. В PyTorch мы используем тензоры для кодирования входных и выходных данных модели, а также параметров модели.

In [150]:
import torch
import numpy as np

### Создание тензоров

Тензор можно создать напрямую из каких-то данных - нам подходят все списки с числами:

In [151]:
some_data = [1, 2, 3, 4]
some_tensor = torch.tensor(some_data)

some_tensor

tensor([1, 2, 3, 4])

In [152]:
some_data = [[1, 2], [3, 4], [5, 6]]
some_tensor = torch.tensor(some_data)

some_tensor

tensor([[1, 2],
        [3, 4],
        [5, 6]])

In [153]:
some_data = [[[1], [2]], [[3], [4]], [[5], [6]]]
some_tensor = torch.tensor(some_data)

some_tensor

tensor([[[1],
         [2]],

        [[3],
         [4]],

        [[5],
         [6]]])

На самом деле про "все" списки с числами - обман. Если у вашего списка есть какой-то уровень вложенности, то должны совпадать размерности у всех вложенных списков (подробнее про размерности поговорим позже):

In [154]:
some_other_data = [[1, 2], [3, 4], [5, 6, 7]]
some_other_tensor = torch.tensor(some_other_data)

some_other_tensor

ValueError: expected sequence of length 2 at dim 1 (got 3)

Также тензоры можно создавать из numpy массивов и наоборот:

In [155]:
some_numpy_array = np.array(some_data)

some_numpy_array

array([[[1],
        [2]],

       [[3],
        [4]],

       [[5],
        [6]]])

In [156]:
some_tensor_from_numpy = torch.from_numpy(some_numpy_array)

some_tensor_from_numpy

tensor([[[1],
         [2]],

        [[3],
         [4]],

        [[5],
         [6]]])

При этом если мы создаем тензор из numpy массива с помощью `torch.from_numpy`, то они делят между собой память, где лежат их данные и, соответственно, при изменении тензора меняется numpy массив и наоборот:

In [157]:
x = np.ones(10)
y = torch.from_numpy(x)

x, y

(array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]),
 tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.], dtype=torch.float64))

In [158]:
x += 1

x, y

(array([2., 2., 2., 2., 2., 2., 2., 2., 2., 2.]),
 tensor([2., 2., 2., 2., 2., 2., 2., 2., 2., 2.], dtype=torch.float64))

In [159]:
x = torch.ones(10)
y = x.numpy()

x, y

(tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]),
 array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.], dtype=float32))

In [160]:
x = x + 1

x, y

(tensor([2., 2., 2., 2., 2., 2., 2., 2., 2., 2.]),
 array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.], dtype=float32))

Какие еще структуры данных в питоне видут себя таким же образом?

In [161]:
a = [1, 2, 3]
b = a
a[0] = 4

a, b

([4, 2, 3], [4, 2, 3])

In [162]:
a = {'a': 1, 'b': 2, 'c': 3}
b = a
a['a'] = 4

a, b

({'a': 4, 'b': 2, 'c': 3}, {'a': 4, 'b': 2, 'c': 3})

Можем создать тензор со случайными или константными значениями:

In [163]:
shape = (2, 3)

random_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)
empty_tensor = torch.empty(shape)

random_tensor, ones_tensor, zeros_tensor, empty_tensor

(tensor([[0.9162, 0.7569, 0.5188],
         [0.9309, 0.2063, 0.9716]]),
 tensor([[1., 1., 1.],
         [1., 1., 1.]]),
 tensor([[0., 0., 0.],
         [0., 0., 0.]]),
 tensor([[1.0781e+02, 4.5764e-41, 1.8829e-33],
         [0.0000e+00, 4.4842e-44, 0.0000e+00]]))

Теперь поговорим про размерности подробнее.

У тензора есть какой-то размер, какая форма. Первое с чем нужно определиться, какой **размерности** тензор - количество осей у него.

In [164]:
shape = (10)  # одна ось (вектор)

tensor = torch.rand(shape)

tensor

tensor([0.9886, 0.9776, 0.2664, 0.2351, 0.5316, 0.6557, 0.7508, 0.7034, 0.9257,
        0.3747])

In [165]:
shape = (2, 3)  # две оси (матрица)

tensor = torch.rand(shape)

tensor

tensor([[0.3360, 0.6676, 0.6393],
        [0.2083, 0.5484, 0.1204]])

In [166]:
shape = (3, 2, 3)  # три оси (и больше - тензор)

tensor = torch.rand(shape)

tensor

tensor([[[0.3533, 0.3038, 0.9383],
         [0.0499, 0.2048, 0.0107]],

        [[0.6261, 0.1152, 0.6618],
         [0.1527, 0.1797, 0.2446]],

        [[0.5750, 0.9594, 0.9684],
         [0.4622, 0.0103, 0.3348]]])

Тензор с размерностью 1 - это просто вектор, список чисел.

Тензор с размерностью 2 - это просто матрица, то есть список списков чисел.

Тензор с размерностью 3 и больше - это тензор, то есть список списков списков ... чисел.

![](image.png)

Получить доступ к размеру уже созданного тензора - метод `.shape`:

In [167]:
some_data = [[[1], [2]], [[3], [4]], [[5], [6]]]
some_tensor = torch.tensor(some_data)

print(some_tensor)
print(some_tensor.shape)

tensor([[[1],
         [2]],

        [[3],
         [4]],

        [[5],
         [6]]])
torch.Size([3, 2, 1])


Давайте сделаем тензор, который будет нам имитировать изображение - сделаем его размер `(c, h, w)`, где `h` и `w` это его высота и ширина, а `c` - число каналов в цветовом пространстве (в черно-белом 1, в RGB 3):

In [168]:
h = 9
w = 16
c = 3

shape = (c, h, w)

image_tensor = torch.rand(shape)

image_tensor

tensor([[[3.0944e-01, 3.9365e-01, 6.9104e-01, 7.5162e-01, 7.2169e-01,
          6.4283e-01, 2.9759e-01, 4.9470e-01, 5.6363e-01, 2.4460e-01,
          4.5527e-01, 4.3759e-01, 2.7594e-02, 8.3896e-01, 4.4945e-01,
          6.7770e-02],
         [8.9559e-01, 2.3096e-01, 6.7702e-01, 9.9586e-01, 5.2315e-01,
          6.4385e-01, 6.9866e-01, 8.7790e-01, 3.4659e-01, 2.1864e-01,
          4.6087e-01, 8.9103e-01, 5.4299e-01, 8.4741e-01, 4.9628e-01,
          6.5341e-01],
         [5.1739e-01, 5.1357e-01, 2.4190e-02, 6.2883e-02, 5.4157e-01,
          7.4747e-01, 8.3588e-01, 5.2169e-01, 7.1076e-01, 7.0912e-01,
          6.8374e-02, 6.0533e-02, 1.1233e-01, 5.0860e-01, 1.8384e-01,
          3.3999e-01],
         [4.2519e-01, 6.9508e-01, 6.1593e-01, 9.6384e-01, 9.5100e-01,
          1.8102e-01, 4.9842e-01, 5.8257e-01, 9.3650e-01, 3.4504e-01,
          3.0350e-01, 1.8327e-01, 8.6843e-01, 3.5835e-01, 3.1274e-01,
          8.3818e-01],
         [9.6008e-01, 4.7893e-01, 1.4755e-01, 9.0739e-01, 3.2900e-01

In [169]:
image_tensor.shape

torch.Size([3, 9, 16])

Можем попробовать поменять размер тензора, например, [вытянуть его в вектор](https://docs.pytorch.org/docs/stable/generated/torch.flatten.html):

In [170]:
image_tensor.flatten()

tensor([3.0944e-01, 3.9365e-01, 6.9104e-01, 7.5162e-01, 7.2169e-01, 6.4283e-01,
        2.9759e-01, 4.9470e-01, 5.6363e-01, 2.4460e-01, 4.5527e-01, 4.3759e-01,
        2.7594e-02, 8.3896e-01, 4.4945e-01, 6.7770e-02, 8.9559e-01, 2.3096e-01,
        6.7702e-01, 9.9586e-01, 5.2315e-01, 6.4385e-01, 6.9866e-01, 8.7790e-01,
        3.4659e-01, 2.1864e-01, 4.6087e-01, 8.9103e-01, 5.4299e-01, 8.4741e-01,
        4.9628e-01, 6.5341e-01, 5.1739e-01, 5.1357e-01, 2.4190e-02, 6.2883e-02,
        5.4157e-01, 7.4747e-01, 8.3588e-01, 5.2169e-01, 7.1076e-01, 7.0912e-01,
        6.8374e-02, 6.0533e-02, 1.1233e-01, 5.0860e-01, 1.8384e-01, 3.3999e-01,
        4.2519e-01, 6.9508e-01, 6.1593e-01, 9.6384e-01, 9.5100e-01, 1.8102e-01,
        4.9842e-01, 5.8257e-01, 9.3650e-01, 3.4504e-01, 3.0350e-01, 1.8327e-01,
        8.6843e-01, 3.5835e-01, 3.1274e-01, 8.3818e-01, 9.6008e-01, 4.7893e-01,
        1.4755e-01, 9.0739e-01, 3.2900e-01, 2.6959e-01, 1.4849e-01, 2.5953e-01,
        1.0842e-01, 2.1139e-01, 4.4521e-

In [171]:
image_tensor.flatten().shape

torch.Size([432])

In [172]:
h * w * c

432

Посчитаем количество элементов в тензоре с помощью [специальной функции](https://pytorch.org/docs/stable/generated/torch.numel.html):

In [173]:
image_tensor.numel()

432

Попробуем поменять размер с помощью функции [reshape](https://pytorch.org/docs/stable/generated/torch.reshape.html#torch.reshape):

In [174]:
image_tensor.reshape(c, h * w).shape

torch.Size([3, 144])

Попробуем собрать из нескольких тензоров один большой:

[torch.cat](https://pytorch.org/docs/stable/generated/torch.cat.html#torch.cat)

In [175]:
x = torch.randn(2, 3)

In [176]:
x

tensor([[-1.3986,  0.2457,  0.2574],
        [-2.6124, -0.3109, -0.1578]])

In [177]:
y = torch.cat((x, x, x), dim=0)
y, y.shape

(tensor([[-1.3986,  0.2457,  0.2574],
         [-2.6124, -0.3109, -0.1578],
         [-1.3986,  0.2457,  0.2574],
         [-2.6124, -0.3109, -0.1578],
         [-1.3986,  0.2457,  0.2574],
         [-2.6124, -0.3109, -0.1578]]),
 torch.Size([6, 3]))

In [178]:
y = torch.stack((x, x, x), dim=0)
y, y.shape

(tensor([[[-1.3986,  0.2457,  0.2574],
          [-2.6124, -0.3109, -0.1578]],
 
         [[-1.3986,  0.2457,  0.2574],
          [-2.6124, -0.3109, -0.1578]],
 
         [[-1.3986,  0.2457,  0.2574],
          [-2.6124, -0.3109, -0.1578]]]),
 torch.Size([3, 2, 3]))

In [179]:
x = torch.randn(3, 3)
y = torch.randn(5, 3)
z = torch.randn(1, 3)

for tensor in [x, y, z]:
    print(tensor)

torch.cat((x, y, z), dim=0)

tensor([[-0.7183, -0.5481, -0.9494],
        [ 0.9284, -0.8161,  0.4858],
        [-0.2189,  0.1857, -0.6343]])
tensor([[-1.1318, -0.7533,  0.5425],
        [ 0.4179,  0.0967,  1.0318],
        [-1.3050, -0.2783, -0.3477],
        [-0.6664, -0.8965,  0.0271],
        [-1.4363,  0.2114,  1.6221]])
tensor([[-0.6601,  1.7837,  0.2563]])


tensor([[-0.7183, -0.5481, -0.9494],
        [ 0.9284, -0.8161,  0.4858],
        [-0.2189,  0.1857, -0.6343],
        [-1.1318, -0.7533,  0.5425],
        [ 0.4179,  0.0967,  1.0318],
        [-1.3050, -0.2783, -0.3477],
        [-0.6664, -0.8965,  0.0271],
        [-1.4363,  0.2114,  1.6221],
        [-0.6601,  1.7837,  0.2563]])

In [180]:
x = torch.randn(2, 3)
y = torch.randn(2, 5)
z = torch.randn(2, 1)

for tensor in [x, y, z]:
    print(tensor)

torch.stack((x, y, z), dim=1)

tensor([[-0.6585, -0.0453,  0.8660],
        [-1.0584,  0.8899, -1.0062]])
tensor([[ 1.1119,  1.9297, -0.2601,  0.3905,  1.2703],
        [ 0.3543,  1.3000, -0.3863, -0.1278,  0.2124]])
tensor([[ 0.8622],
        [-0.7071]])


RuntimeError: stack expects each tensor to be equal size, but got [2, 3] at entry 0 and [2, 5] at entry 1

Теперь добавим дополнительную ось:

[torch.unsqueeze](https://pytorch.org/docs/stable/generated/torch.unsqueeze.html)

In [181]:
x = torch.rand(2, 3)

print(x.shape)
print()
print(x.unsqueeze(0).shape)
print(x[None, :, :].shape)
print()
print(x.unsqueeze(1).shape)
print(x[:, None, :].shape)
print()
print(x.unsqueeze(2).shape)
print(x[:, :, None].shape)

torch.Size([2, 3])

torch.Size([1, 2, 3])
torch.Size([1, 2, 3])

torch.Size([2, 1, 3])
torch.Size([2, 1, 3])

torch.Size([2, 3, 1])
torch.Size([2, 3, 1])


Уберем лишние оси (где размер единичка):

In [182]:
x = torch.rand(1, 2, 1, 3)

print(x.shape)
print()
print(x.squeeze().shape)
print(x[0, :, 0, :].shape)
print()
print(x.squeeze(1).shape)

torch.Size([1, 2, 1, 3])

torch.Size([2, 3])
torch.Size([2, 3])

torch.Size([1, 2, 1, 3])


Теперь поговорим про типы данных в тензорах. По умолчанию в тензорах лежат числа в torch.float32 для вещественных и torch.int64 для целочисленных.

In [183]:
tensor = torch.tensor([1.5, 2.2, 3.7, 4.9])

tensor

tensor([1.5000, 2.2000, 3.7000, 4.9000])

In [184]:
tensor.dtype

torch.float32

In [185]:
tensor = torch.tensor([1.5, 2.2, 3.7, 4.9], dtype=torch.float16)

tensor

tensor([1.5000, 2.1992, 3.6992, 4.8984], dtype=torch.float16)

In [186]:
tensor = torch.tensor([1.5, 2.2, 3.7, 4.9], dtype=torch.float64)

tensor

tensor([1.5000, 2.2000, 3.7000, 4.9000], dtype=torch.float64)

In [187]:
tensor = torch.tensor([15, 22, 37, 49])

tensor

tensor([15, 22, 37, 49])

In [188]:
tensor.dtype

torch.int64

In [189]:
tensor = torch.tensor([15, 22, 37, 49], dtype=torch.int32)

tensor

tensor([15, 22, 37, 49], dtype=torch.int32)

In [190]:
tensor = torch.tensor([15, 22, 37, 49], dtype=torch.int16)

tensor

tensor([15, 22, 37, 49], dtype=torch.int16)

Размещение тензора на GPU:

In [191]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

True
NVIDIA H100 80GB HBM3


In [192]:
! nvidia-smi

Sat Oct 18 14:52:14 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.133.20             Driver Version: 570.133.20     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:3B:00.0 Off |                    0 |
| N/A   37C    P0            121W /  700W |     619MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [193]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print(device)

cuda:0


In [194]:
tensor = torch.tensor([15, 22, 37, 49], device=device)

tensor

tensor([15, 22, 37, 49], device='cuda:0')

In [195]:
tensor = torch.tensor([15, 22, 37, 49])

print(tensor)

tensor = tensor.to(device)

tensor

tensor([15, 22, 37, 49])


tensor([15, 22, 37, 49], device='cuda:0')

In [196]:
tensor.to(torch.int32)

tensor([15, 22, 37, 49], device='cuda:0', dtype=torch.int32)

In [197]:
tensor = tensor.cpu()

tensor

tensor([15, 22, 37, 49])

In [198]:
tensor.cuda() # звучит некрасиво

tensor([15, 22, 37, 49], device='cuda:0')

In [199]:
a = torch.rand(2, 3)
b = torch.rand(2, 3)

a + b

tensor([[1.0134, 1.1348, 0.6490],
        [0.4292, 0.7326, 1.6920]])

In [200]:
device_a = a.to(device)

device_a + b

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

In [201]:
device_b = b.to(device)

device_a + device_b

tensor([[1.0134, 1.1348, 0.6490],
        [0.4292, 0.7326, 1.6920]], device='cuda:0')

### Операции с тензорами

Большая часть операций с тензорами хорошо описана в их [документации](https://pytorch.org/docs/stable/torch.html), разберем основные:

In [202]:
a = torch.rand(2, 3)
b = torch.rand(2, 3)

a, b

(tensor([[0.3023, 0.3160, 0.8961],
         [0.2425, 0.1270, 0.4220]]),
 tensor([[0.9758, 0.6088, 0.5691],
         [0.4377, 0.8124, 0.2522]]))

In [203]:
# поэлементные

print(a + b)

print()

print(torch.add(a, b))

print()

print(a.add(b))

tensor([[1.2781, 0.9248, 1.4652],
        [0.6802, 0.9394, 0.6742]])

tensor([[1.2781, 0.9248, 1.4652],
        [0.6802, 0.9394, 0.6742]])

tensor([[1.2781, 0.9248, 1.4652],
        [0.6802, 0.9394, 0.6742]])


In [204]:
print(a - b)

print()

print(torch.sub(a, b))

print()

print(a.sub(b))

tensor([[-0.6734, -0.2928,  0.3270],
        [-0.1953, -0.6855,  0.1698]])

tensor([[-0.6734, -0.2928,  0.3270],
        [-0.1953, -0.6855,  0.1698]])

tensor([[-0.6734, -0.2928,  0.3270],
        [-0.1953, -0.6855,  0.1698]])


In [205]:
print(a * b)

print()

print(torch.mul(a, b))

print()

print(a.mul(b))

tensor([[0.2950, 0.1924, 0.5099],
        [0.1061, 0.1031, 0.1064]])

tensor([[0.2950, 0.1924, 0.5099],
        [0.1061, 0.1031, 0.1064]])

tensor([[0.2950, 0.1924, 0.5099],
        [0.1061, 0.1031, 0.1064]])


In [206]:
print(a / b)

print()

print(torch.div(a, b))

print()

print(a.div(b))

tensor([[0.3098, 0.5190, 1.5746],
        [0.5540, 0.1563, 1.6734]])

tensor([[0.3098, 0.5190, 1.5746],
        [0.5540, 0.1563, 1.6734]])

tensor([[0.3098, 0.5190, 1.5746],
        [0.5540, 0.1563, 1.6734]])


In [207]:
a = torch.rand(2, 3)
b = torch.rand(3, 4)
c = torch.rand(5, 5)

a, b, c

(tensor([[0.8390, 0.6181, 0.5556],
         [0.5688, 0.1916, 0.8714]]),
 tensor([[0.7626, 0.1089, 0.0345, 0.3713],
         [0.8322, 0.2538, 0.9634, 0.2486],
         [0.0483, 0.7199, 0.1978, 0.4212]]),
 tensor([[0.3440, 0.4603, 0.0643, 0.0414, 0.7149],
         [0.4433, 0.1984, 0.4873, 0.0436, 0.6327],
         [0.7375, 0.9849, 0.0927, 0.4394, 0.7713],
         [0.9582, 0.1443, 0.7125, 0.7064, 0.8014],
         [0.1553, 0.8568, 0.2822, 0.1922, 0.5993]]))

In [208]:
# матричные операции

print(a @ b, (a @ b).shape)

print()

print(torch.matmul(a, b), torch.matmul(a, b).shape)

print()

print(c.trace())

tensor([[1.1811, 0.6483, 0.7343, 0.6992],
        [0.6354, 0.7379, 0.3766, 0.6258]]) torch.Size([2, 4])

tensor([[1.1811, 0.6483, 0.7343, 0.6992],
        [0.6354, 0.7379, 0.3766, 0.6258]]) torch.Size([2, 4])

tensor(1.9408)


### [Автоматическое дифференцирование](https://pytorch.org/docs/stable/notes/autograd.html)

$y = x_1^2 + 3(x_2x_3 - x_1^2)$

<img src="image2.png" width="600" />

Как в торче строить подобные графы и считать градиенты?

In [209]:
x = torch.rand(5)

x

tensor([0.9236, 0.1689, 0.7670, 0.4427, 0.6596])

In [210]:
w = torch.rand(3, 5, requires_grad=True)

w

tensor([[0.8900, 0.6358, 0.0849, 0.3495, 0.5992],
        [0.0782, 0.5831, 0.2315, 0.4006, 0.6789],
        [0.0976, 0.1790, 0.7281, 0.0055, 0.8441]], requires_grad=True)

In [211]:
print(w.grad)

None


In [212]:
loss = torch.mean(w @ x)
loss.backward()

In [213]:
print(w.grad)

tensor([[0.3079, 0.0563, 0.2557, 0.1476, 0.2199],
        [0.3079, 0.0563, 0.2557, 0.1476, 0.2199],
        [0.3079, 0.0563, 0.2557, 0.1476, 0.2199]])


Построим граф из примера и проверим, что нас не обманывают.

In [214]:
x1 = torch.tensor([1.], requires_grad=True)
x2 = torch.tensor([2.], requires_grad=True)
x3 = torch.tensor([3.], requires_grad=True)

y = x1 ** 2 + 3 * (x2 * x3 - x1 ** 2)

y.backward()

print(x1.grad)
print(x2.grad)
print(x3.grad)

tensor([-4.])
tensor([9.])
tensor([6.])


Интересный факт - у промежуточных переменных градиенты не сохраняются.

In [215]:
x1 = torch.tensor([1.], requires_grad=True)
x2 = torch.tensor([2.], requires_grad=True)
x3 = torch.tensor([3.], requires_grad=True)

z1 = x1 **2

y = z1 + 3 * (x2 * x3 - z1)

y.backward()

print(x1.grad)
print(x2.grad)
print(x3.grad)
print(z1.grad)

tensor([-4.])
tensor([9.])
tensor([6.])
None


/tmp/ipykernel_986678/2838312020.py:14: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at aten/src/ATen/core/TensorBody.h:489.)
  print(z1.grad)


При инференсе модели мы не хотим считать градиенты. Отключать их подсчет можно при помощи следующих контекстных менеджеров и декораторов.

In [216]:
a = torch.rand(3, 5, requires_grad=True)
b = torch.rand(3, 5, requires_grad=True)

with torch.inference_mode():
    loss = torch.sum(a - b)

loss

tensor(-0.4206)

In [217]:
loss.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [218]:
a = torch.rand(3, 5, requires_grad=True)
b = torch.rand(3, 5, requires_grad=True)

with torch.no_grad():
    loss = torch.sum(a - b)

loss

tensor(-1.9602)

In [219]:
loss.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [220]:
a = torch.rand(3, 5, requires_grad=True)
b = torch.rand(3, 5, requires_grad=True)

loss = torch.mean(a + b)
loss

tensor(0.8926, grad_fn=<MeanBackward0>)

In [221]:
@torch.no_grad()
def foo():
    a = torch.rand(3, 5, requires_grad=True)
    b = torch.rand(3, 5, requires_grad=True)
    
    loss = torch.mean(a + b)
    
    print(f'{loss=}')
    

In [222]:
foo()

loss=tensor(0.9969)


In [223]:
@torch.inference_mode()
def foo():
    a = torch.rand(3, 5, requires_grad=True)
    b = torch.rand(3, 5, requires_grad=True)
    
    loss = torch.mean(a + b)
    
    print(f'{loss=}')

In [224]:
foo()

loss=tensor(1.1894)


## Полносвязные слои и функции активации в `PyTorch`

In [225]:
from torch import nn # neural networks

### Полносвязный слой

>$y_j = \sum\limits_{i=1}^{n}x_iw_{ji} + b_j$

[torch.nn.linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)


In [226]:
layer = nn.Linear(in_features=5, out_features=3)

In [227]:
layer

Linear(in_features=5, out_features=3, bias=True)

In [228]:
layer.weight

Parameter containing:
tensor([[ 0.0513, -0.0408,  0.3426, -0.3907, -0.2041],
        [-0.2021,  0.3864, -0.4326,  0.0444, -0.4332],
        [-0.0147,  0.1667, -0.1046,  0.4341,  0.0014]], requires_grad=True)

In [229]:
layer.weight.shape

torch.Size([3, 5])

In [230]:
layer.bias

Parameter containing:
tensor([-0.2280,  0.1153,  0.2326], requires_grad=True)

In [231]:
layer = nn.Linear(in_features=5, out_features=3, bias=False)

In [232]:
print(layer.bias)

None


In [233]:
layer.__call__

<bound method Module._wrapped_call_impl of Linear(in_features=5, out_features=3, bias=False)>

In [234]:
x = torch.randn(5)

print(layer(x))

tensor([ 0.6474, -0.0336, -0.8819], grad_fn=<SqueezeBackward4>)


In [235]:
layer1 = nn.Linear(in_features=5, out_features=3)
layer2 = nn.Linear(in_features=3, out_features=1)

layer2(layer1(x))

tensor([-0.7833], grad_fn=<ViewBackward0>)

### Функции активации

<img src="image_sigmoid.png" width="600" />

In [236]:
activation = nn.Sigmoid()

In [237]:
x = torch.randn(5)

print(x)

print(activation(x))

tensor([ 1.1218, -0.4029,  0.3856,  1.4492, -1.4418])
tensor([0.7543, 0.4006, 0.5952, 0.8099, 0.1913])


> ReLU

<img src="relu.png" width="600" />

In [238]:
activation = nn.ReLU()

In [239]:
x = torch.randn(5)

print(x)

print(activation(x))

tensor([-0.1644, -0.1270, -0.2620,  1.6631, -1.2930])
tensor([0.0000, 0.0000, 0.0000, 1.6631, 0.0000])


> Leaky ReLU 

<img src="leaky_relu.png" width="600" />

In [240]:
activation = nn.LeakyReLU(negative_slope=0.001)

In [241]:
x = torch.randn(5)

print(x)

print(activation(x))

tensor([ 1.3890, -0.7775, -0.2976,  0.1562, -0.1113])
tensor([ 1.3890e+00, -7.7753e-04, -2.9764e-04,  1.5618e-01, -1.1129e-04])


In [242]:
layer1 = nn.Linear(in_features=5, out_features=3)
activation = nn.LeakyReLU(negative_slope=0.001)
layer2 = nn.Linear(in_features=3, out_features=1)

layer2(activation(layer1(x)))

tensor([-0.5740], grad_fn=<ViewBackward0>)

## Градиентный спуск своими руками

Сделаем датасет своими руками и попробуем что-нибудь обучить.

In [243]:
n_features = 2
n_objects = 300

torch.manual_seed(0)

w_true = torch.randn(n_features)
b_true = torch.randn(1)

x = (torch.rand(n_objects, n_features) - 0.5) * 10 * (torch.arange(n_features) * 2 + 1)
y = torch.matmul(x, w_true) + torch.randn(n_objects) + b_true

In [244]:
x.shape

torch.Size([300, 2])

In [245]:
y.shape

torch.Size([300])

In [ ]:
n_steps = 200
step_size = 1e-2

In [246]:
w = torch.rand(n_features, requires_grad=True)
b = torch.rand(1, requires_grad=True)

for i in range(n_steps):
    y_pred = torch.matmul(x, w) + b

    mse = torch.mean((y_pred - y) ** 2)

    if i < 20 or i % 10 == 0:
        print(f'MSE на шаге {i + 1} {mse.item():.5f}')

    mse.backward()

    with torch.no_grad():
        w -= w.grad * step_size
        b -= b.grad * step_size

    w.grad.zero_()
    b.grad.zero_()

MSE на шаге 1 23.65750
MSE на шаге 2 22.81619
MSE на шаге 3 22.04544
MSE на шаге 4 21.33889
MSE на шаге 5 20.69075
MSE на шаге 6 20.09575
MSE на шаге 7 19.54912
MSE на шаге 8 19.04651
MSE на шаге 9 18.58397
MSE на шаге 10 18.15789
MSE на шаге 11 17.76502
MSE на шаге 12 17.40238
MSE на шаге 13 17.06727
MSE на шаге 14 16.75723
MSE на шаге 15 16.47004
MSE на шаге 16 16.20366
MSE на шаге 17 15.95624
MSE на шаге 18 15.72611
MSE на шаге 19 15.51174
MSE на шаге 20 15.31174
MSE на шаге 21 15.12485
MSE на шаге 31 13.76920
MSE на шаге 41 12.93688
MSE на шаге 51 12.33191
MSE на шаге 61 11.83776
MSE на шаге 71 11.40731
MSE на шаге 81 11.02038
MSE на шаге 91 10.66741
MSE на шаге 101 10.34304
MSE на шаге 111 10.04376
MSE на шаге 121 9.76687
MSE на шаге 131 9.51013
MSE на шаге 141 9.27160
MSE на шаге 151 9.04955
MSE на шаге 161 8.84243
MSE на шаге 171 8.64885
MSE на шаге 181 8.46756
MSE на шаге 191 8.29742
MSE на шаге 201 8.13741
MSE на шаге 211 7.98662
MSE на шаге 221 7.84422
MSE на шаге 231 7.70944

In [247]:
layer = nn.Linear(in_features=n_features, out_features=1)


for i in range(n_steps):
    y_pred = layer(x)

    mse = torch.mean((y_pred - y) ** 2)
    
    if i < 20 or i % 10 == 0:
        print(f'MSE на шаге {i + 1} {mse.item():.5f}')

    mse.backward()

    with torch.no_grad():
        layer.weight -= layer.weight.grad * step_size
        layer.bias -= layer.bias.grad * step_size
    
    layer.zero_grad()  # == layer.weight.grad.zero_() + layer.bias.grad.zero_()

MSE на шаге 1 61.04560
MSE на шаге 2 58.94495
MSE на шаге 3 57.03117
MSE на шаге 4 55.28730
MSE на шаге 5 53.69790
MSE на шаге 6 52.24896
MSE на шаге 7 50.92775
MSE на шаге 8 49.72266
MSE на шаге 9 48.62318
MSE на шаге 10 47.61973
MSE на шаге 11 46.70360
MSE на шаге 12 45.86688
MSE на шаге 13 45.10239
MSE на шаге 14 44.40358
MSE на шаге 15 43.76451
MSE на шаге 16 43.17978
MSE на шаге 17 42.64446
MSE на шаге 18 42.15411
MSE на шаге 19 41.70465
MSE на шаге 20 41.29240
MSE на шаге 21 40.91401
MSE на шаге 31 38.44008
MSE на шаге 41 37.26230
MSE на шаге 51 36.60081
MSE на шаге 61 36.15357
MSE на шаге 71 35.80275
MSE на шаге 81 35.50163
MSE на шаге 91 35.23107
MSE на шаге 101 34.98260
MSE на шаге 111 34.75201
MSE на шаге 121 34.53681
MSE на шаге 131 34.33526
MSE на шаге 141 34.14599
MSE на шаге 151 33.96782
MSE на шаге 161 33.79974
MSE на шаге 171 33.64083
MSE на шаге 181 33.49028
MSE на шаге 191 33.34734
MSE на шаге 201 33.21135
MSE на шаге 211 33.08170
MSE на шаге 221 32.95787
MSE на шаге 

In [248]:
print(layer(x).shape)
print(y.shape)
print((layer(x) - y).shape)

torch.Size([300, 1])
torch.Size([300])
torch.Size([300, 300])


In [249]:
print(layer(x).flatten().shape)
print(y.shape)
print((layer(x).flatten() - y).shape)

torch.Size([300])
torch.Size([300])
torch.Size([300])


In [250]:
layer = nn.Linear(in_features=n_features, out_features=1)

for i in range(n_steps):
    y_pred = layer(x).flatten()

    mse = torch.mean((y_pred - y) ** 2)
    
    if i < 20 or i % 10 == 0:
        print(f'MSE на шаге {i + 1} {mse.item():.5f}')

    mse.backward()

    with torch.no_grad():
        layer.weight -= layer.weight.grad * step_size
        layer.bias -= layer.bias.grad * step_size

    layer.zero_grad()


MSE на шаге 1 24.25042
MSE на шаге 2 22.88535
MSE на шаге 3 21.63854
MSE на шаге 4 20.49928
MSE на шаге 5 19.45785
MSE на шаге 6 18.50541
MSE на шаге 7 17.63392
MSE на шаге 8 16.83607
MSE на шаге 9 16.10523
MSE на шаге 10 15.43534
MSE на шаге 11 14.82091
MSE на шаге 12 14.25697
MSE на шаге 13 13.73897
MSE на шаге 14 13.26277
MSE на шаге 15 12.82464
MSE на шаге 16 12.42116
MSE на шаге 17 12.04922
MSE на шаге 18 11.70602
MSE на шаге 19 11.38899
MSE на шаге 20 11.09580
MSE на шаге 21 10.82432
MSE на шаге 31 8.95535
MSE на шаге 41 7.93718
MSE на шаге 51 7.27486
MSE на шаге 61 6.77494
MSE на шаге 71 6.36042
MSE на шаге 81 5.99940
MSE на шаге 91 5.67757
MSE на шаге 101 5.38761
MSE на шаге 111 5.12505
MSE на шаге 121 4.88670
MSE на шаге 131 4.67001
MSE на шаге 141 4.47277
MSE на шаге 151 4.29308
MSE на шаге 161 4.12922
MSE на шаге 171 3.97963
MSE на шаге 181 3.84294
MSE на шаге 191 3.71790
MSE на шаге 201 3.60337
MSE на шаге 211 3.49836
MSE на шаге 221 3.40193
MSE на шаге 231 3.31326
MSE на ш

In [251]:
n_features = 5
n_objects = 300

torch.manual_seed(0)

w_true = torch.randn(n_features)

x = (torch.rand(n_objects, n_features) - 0.5) * 10 * (torch.arange(n_features) * 2 + 1)
y = torch.matmul(x, w_true) + torch.randn(n_objects)

In [252]:
x.shape, y.shape

(torch.Size([300, 5]), torch.Size([300]))

In [253]:
n_steps = 500
step_size = 1e-3

In [254]:
layer = nn.Linear(in_features=n_features, out_features=1)

for i in range(n_steps):
    y_pred = layer(x).flatten()

    mse = torch.mean((y_pred - y) ** 2)
    
    if i < 20 or i % 50 == 0:
        print(f'MSE на шаге {i + 1} {mse.item():.5f}')

    mse.backward()

    with torch.no_grad():
        layer.weight -= layer.weight.grad * step_size
        layer.bias -= layer.bias.grad * step_size

    layer.zero_grad()

MSE на шаге 1 1522.64282
MSE на шаге 2 345.22870
MSE на шаге 3 134.71320
MSE на шаге 4 69.14159
MSE на шаге 5 45.61798
MSE на шаге 6 36.15922
MSE на шаге 7 31.74179
MSE на шаге 8 29.24492
MSE на шаге 9 27.53952
MSE на шаге 10 26.19924
MSE на шаге 11 25.05364
MSE на шаге 12 24.02866
MSE на шаге 13 23.08827
MSE на шаге 14 22.21260
MSE на шаге 15 21.38941
MSE на шаге 16 20.61047
MSE на шаге 17 19.86992
MSE на шаге 18 19.16344
MSE на шаге 19 18.48770
MSE на шаге 20 17.84015
MSE на шаге 51 6.27402
MSE на шаге 101 1.74083
MSE на шаге 151 1.02593
MSE на шаге 201 0.91269
MSE на шаге 251 0.89435
MSE на шаге 301 0.89104
MSE на шаге 351 0.89019
MSE на шаге 401 0.88977
MSE на шаге 451 0.88948


In [255]:
n_steps = 1000
step_size = 3e-4

In [256]:
layer1 = nn.Linear(in_features=n_features, out_features=3)
layer2 = nn.Linear(in_features=3, out_features=1)
activation = nn.ReLU()


for i in range(n_steps):
    y_pred = layer2(activation(layer1(x))).flatten()

    mse = torch.mean((y_pred - y) ** 2)

    if i < 20 or i % 50 == 0:
        print(f'MSE на шаге {i + 1} {mse.item():.5f}')

    mse.backward()

    with torch.no_grad():
        layer1.weight -= layer1.weight.grad * step_size
        layer1.bias -= layer1.bias.grad * step_size
        layer2.weight -= layer2.weight.grad * step_size
        layer2.bias -= layer2.bias.grad * step_size

    layer1.zero_grad()
    layer2.zero_grad()

MSE на шаге 1 2065.53369
MSE на шаге 2 1941.40039
MSE на шаге 3 1890.73816
MSE на шаге 4 1863.17334
MSE на шаге 5 1843.74536
MSE на шаге 6 1825.21838
MSE на шаге 7 1803.15234
MSE на шаге 8 1775.96960
MSE на шаге 9 1741.44421
MSE на шаге 10 1696.50024
MSE на шаге 11 1638.66479
MSE на шаге 12 1570.60828
MSE на шаге 13 1498.17651
MSE на шаге 14 1426.69543
MSE на шаге 15 1361.06189
MSE на шаге 16 1304.04346
MSE на шаге 17 1252.44177
MSE на шаге 18 1201.12524
MSE на шаге 19 1142.44934
MSE на шаге 20 1069.69470
MSE на шаге 51 10.77816
MSE на шаге 101 5.31390
MSE на шаге 151 3.22006
MSE на шаге 201 2.08225
MSE на шаге 251 1.45981
MSE на шаге 301 1.16964
MSE на шаге 351 1.03444
MSE на шаге 401 0.97123
MSE на шаге 451 0.94165
MSE на шаге 501 0.92668
MSE на шаге 551 0.91742
MSE на шаге 601 0.91137
MSE на шаге 651 0.90724
MSE на шаге 701 0.90359
MSE на шаге 751 0.90093
MSE на шаге 801 0.89879
MSE на шаге 851 0.89742
MSE на шаге 901 0.89643
MSE на шаге 951 0.89556
